In [ ]:
# %%
import time
import urllib.request
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_SEED = 42

FIGURE_DIR = Path("./figures")
FIGURE_DIR.mkdir(exist_ok=True)

TRAIN_URL = (
    "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt"
)
TEST_URL = (
    "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt"
)

COLUMN_NAMES = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty_level",
]

CATEGORICAL_COLUMNS = ["protocol_type", "service", "flag"]

ATTACK_MAP = {
    "normal": "Normal",
    "neptune": "DoS",
    "back": "DoS",
    "land": "DoS",
    "pod": "DoS",
    "smurf": "DoS",
    "teardrop": "DoS",
    "mailbomb": "DoS",
    "apache2": "DoS",
    "processtable": "DoS",
    "udpstorm": "DoS",
    "worm": "DoS",
    "satan": "Probe",
    "ipsweep": "Probe",
    "nmap": "Probe",
    "portsweep": "Probe",
    "mscan": "Probe",
    "saint": "Probe",
    "guess_passwd": "R2L",
    "ftp_write": "R2L",
    "imap": "R2L",
    "phf": "R2L",
    "multihop": "R2L",
    "warezmaster": "R2L",
    "warezclient": "R2L",
    "spy": "R2L",
    "xlock": "R2L",
    "xsnoop": "R2L",
    "snmpguess": "R2L",
    "snmpgetattack": "R2L",
    "httptunnel": "R2L",
    "sendmail": "R2L",
    "named": "R2L",
    "buffer_overflow": "U2R",
    "loadmodule": "U2R",
    "rootkit": "U2R",
    "perl": "U2R",
    "sqlattack": "U2R",
    "xterm": "U2R",
    "ps": "U2R",
}

MINORITY_CLASSES = ("U2R", "R2L")

plt.rcParams.update({"font.size": 12, "axes.grid": True, "grid.alpha": 0.25})
FIGSIZE = (15, 15 * 9 / 16)


# %%
# =========================================================
# DATA LOADING
# =========================================================
def load_nsl_kdd(max_retries=3, timeout=60):
    def fetch(url):
        last_error = None
        for attempt in range(1, max_retries + 1):
            try:
                return (
                    urllib.request.urlopen(url, timeout=timeout).read().decode("utf-8")
                )
            except Exception as e:
                last_error = e
                print(f"Attempt {attempt}/{max_retries} failed for {url}: {e}")
        raise last_error

    train_text = fetch(TRAIN_URL)
    test_text = fetch(TEST_URL)
    train_df = pd.read_csv(StringIO(train_text), names=COLUMN_NAMES)
    test_df = pd.read_csv(StringIO(test_text), names=COLUMN_NAMES)
    return train_df, test_df


def prepare_labels(df):
    df = df.copy()
    df["target"] = (df["label"] != "normal").astype(int)
    df["category"] = df["label"].map(ATTACK_MAP).fillna("Unknown")
    df = df.drop(columns=["label", "difficulty_level"])
    return df


def load_and_prepare_dataset():
    train_full_df, test_df = load_nsl_kdd()
    train_full_df = prepare_labels(train_full_df)
    test_df = prepare_labels(test_df)

    print(f"Original train: {len(train_full_df):,} | Original test: {len(test_df):,}")
    print("Train category distribution:")
    print(train_full_df["category"].value_counts(normalize=True).mul(100).round(2))
    print("\nTest category distribution (note: may include unseen attack types):")
    print(test_df["category"].value_counts(normalize=True).mul(100).round(2))

    train_labels = set(train_full_df["category"].unique())
    test_labels = set(test_df["category"].unique())
    novel = test_labels - train_labels
    if novel:
        print(f"\nNovel attack categories in test only: {novel}")

    return train_full_df, test_df


def split_raw_data(train_full_df, test_df, random_state=RANDOM_SEED):
    train_df, val_df = train_test_split(
        train_full_df,
        test_size=0.20,
        stratify=train_full_df["target"],
        random_state=random_state,
    )
    return train_df, val_df, test_df


# ---- execution ----
train_full_df, test_df = load_and_prepare_dataset()
train_df, val_df, test_df = split_raw_data(
    train_full_df, test_df, random_state=RANDOM_SEED
)


# %%
# =========================================================
# ENCODING / LABELS
# =========================================================
def fit_encoders(train_df):
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    ohe.fit(train_df[CATEGORICAL_COLUMNS])

    label_encoder = LabelEncoder()
    label_encoder.fit(train_df["category"])

    return ohe, label_encoder


def transform_features_and_labels(df, ohe, label_encoder):
    df = df.copy()

    cat_encoded = ohe.transform(df[CATEGORICAL_COLUMNS])
    cat_cols = ohe.get_feature_names_out(CATEGORICAL_COLUMNS)
    cat_df = pd.DataFrame(cat_encoded, columns=cat_cols, index=df.index)

    numeric_df = df.drop(columns=CATEGORICAL_COLUMNS + ["target", "category"])
    x = pd.concat([numeric_df, cat_df], axis=1)

    known_labels = set(label_encoder.classes_)
    known_mask = df["category"].isin(known_labels)

    y_multiclass = pd.Series(-1, index=df.index, dtype=int)
    y_multiclass[known_mask] = label_encoder.transform(df.loc[known_mask, "category"])

    y_binary = df["target"]

    return x, y_binary, y_multiclass


def assign_unknown_class(y_mc_splits, class_names):
    unknown_idx = len(class_names)
    class_names_with_unknown = np.append(class_names, "Unknown")

    for split_name, y_mc in y_mc_splits.items():
        n_novel = (y_mc == -1).sum()
        print(f"Novel/unseen category rows — {split_name}: {n_novel}")

    y_mc_splits = {
        split_name: y_mc.replace(-1, unknown_idx)
        for split_name, y_mc in y_mc_splits.items()
    }

    return y_mc_splits, class_names_with_unknown


# ---- execution ----
ohe, label_encoder = fit_encoders(train_df)
class_names = label_encoder.classes_

x_tr, y_tr, y_tr_mc = transform_features_and_labels(train_df, ohe, label_encoder)
x_val, y_val, y_val_mc = transform_features_and_labels(val_df, ohe, label_encoder)
x_test, y_test, y_test_mc = transform_features_and_labels(test_df, ohe, label_encoder)

print(f"Train: {x_tr.shape} | Val: {x_val.shape} | Test: {x_test.shape}")

y_mc_splits, class_names_with_unknown = assign_unknown_class(
    {"Train": y_tr_mc, "Val": y_val_mc, "Test": y_test_mc}, class_names
)
y_tr_mc, y_val_mc, y_test_mc = (
    y_mc_splits["Train"],
    y_mc_splits["Val"],
    y_mc_splits["Test"],
)

x_tr_mc, x_val_mc, x_test_mc = x_tr, x_val, x_test


# %%
# =========================================================
# SCALING + PCA
# =========================================================
def fast_pca_fit(x, n_components=None):
    x = np.asarray(x, dtype=np.float64)
    n_samples, n_features = x.shape

    mean = x.mean(axis=0)
    x_centered = x - mean

    cov = (x_centered.T @ x_centered) / (n_samples - 1)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)

    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = np.clip(eigenvalues[order], 0, None)
    eigenvectors = eigenvectors[:, order]

    idx = np.argmax(np.abs(eigenvectors), axis=0)
    signs = np.sign(eigenvectors[idx, np.arange(eigenvectors.shape[1])])
    eigenvectors = eigenvectors * signs

    ratio = eigenvalues / eigenvalues.sum()

    if n_components is None:
        k = n_features
    elif isinstance(n_components, float):
        k = int(np.argmax(np.cumsum(ratio) >= n_components)) + 1
    else:
        k = int(n_components)

    components = eigenvectors[:, :k].T
    explained_variance = eigenvalues[:k]
    explained_variance_ratio = ratio[:k]

    return mean, components, explained_variance, explained_variance_ratio


def fast_pca_transform(x, mean, components):
    x = np.asarray(x, dtype=np.float64)
    return (x - mean) @ components.T


def scale_features(x_tr, x_val, x_test):
    scaler = StandardScaler()
    x_tr_scaled = scaler.fit_transform(x_tr.astype(np.float64))
    x_val_scaled = scaler.transform(x_val.astype(np.float64))
    x_test_scaled = scaler.transform(x_test.astype(np.float64))
    return scaler, x_tr_scaled, x_val_scaled, x_test_scaled


def compare_pca_variants(x_tr_scaled, x_val_scaled, x_test_scaled, n_components="mle"):
    t0 = time.perf_counter()
    sklearn_pca = PCA(
        n_components=n_components, svd_solver="full", random_state=RANDOM_SEED
    )
    x_tr_pca_sklearn = sklearn_pca.fit_transform(x_tr_scaled)
    t1 = time.perf_counter()
    x_val_pca_sklearn = sklearn_pca.transform(x_val_scaled)
    x_test_pca_sklearn = sklearn_pca.transform(x_test_scaled)
    sklearn_fit_transform_time = t1 - t0

    t2 = time.perf_counter()
    mean, components, explained_variance, explained_variance_ratio = fast_pca_fit(
        x_tr_scaled, n_components=sklearn_pca.n_components_
    )
    x_tr_pca_fast = fast_pca_transform(x_tr_scaled, mean, components)
    t3 = time.perf_counter()
    x_val_pca_fast = fast_pca_transform(x_val_scaled, mean, components)
    x_test_pca_fast = fast_pca_transform(x_test_scaled, mean, components)
    fast_fit_transform_time = t3 - t2

    all_sklearn = np.vstack([x_tr_pca_sklearn, x_val_pca_sklearn, x_test_pca_sklearn])
    all_fast = np.vstack([x_tr_pca_fast, x_val_pca_fast, x_test_pca_fast])

    signs = np.sign(np.sum(all_sklearn * all_fast, axis=0, keepdims=True))
    signs[signs == 0] = 1
    all_fast_aligned = all_fast * signs

    abs_diff = np.abs(all_sklearn - all_fast_aligned)
    max_abs_diff = abs_diff.max()
    mean_abs_diff = abs_diff.mean()

    print(f"PCA equivalence check:")
    print(f"  n_components (sklearn, MLE): {sklearn_pca.n_components_}")
    print(
        f"  variance retained (sklearn): {sklearn_pca.explained_variance_ratio_.sum():.4f}"
    )
    print(f"  max abs diff (sklearn vs fast): {max_abs_diff:.6e}")
    print(f"  mean abs diff (sklearn vs fast): {mean_abs_diff:.6e}")
    print(f"  sklearn fit_transform time: {sklearn_fit_transform_time:.4f}s")
    print(f"  fast (from-scratch) fit_transform time: {fast_fit_transform_time:.4f}s")

    return {
        "sklearn_pca": sklearn_pca,
        "fast_pca_params": (
            mean,
            components,
            explained_variance,
            explained_variance_ratio,
        ),
        "x_tr_pca_sklearn": x_tr_pca_sklearn,
        "x_val_pca_sklearn": x_val_pca_sklearn,
        "x_test_pca_sklearn": x_test_pca_sklearn,
        "x_tr_pca_fast": x_tr_pca_fast,
        "x_val_pca_fast": x_val_pca_fast,
        "x_test_pca_fast": x_test_pca_fast,
        "equivalence_stats": {
            "max_abs_diff": max_abs_diff,
            "mean_abs_diff": mean_abs_diff,
            "sklearn_time_sec": sklearn_fit_transform_time,
            "fast_time_sec": fast_fit_transform_time,
        },
    }


def fit_multiclass_pca(x_tr_scaled, x_val_scaled, x_test_scaled, n_components="mle"):
    pca_mc = PCA(n_components=n_components, svd_solver="full", random_state=RANDOM_SEED)
    x_tr_mc_pca = pca_mc.fit_transform(x_tr_scaled)
    x_val_mc_pca = pca_mc.transform(x_val_scaled)
    x_test_mc_pca = pca_mc.transform(x_test_scaled)
    return pca_mc, x_tr_mc_pca, x_val_mc_pca, x_test_mc_pca


# ---- execution ----
scaler, x_tr_scaled, x_val_scaled, x_test_scaled = scale_features(x_tr, x_val, x_test)
for name, y in [("Train", y_tr), ("Val", y_val), ("Test", y_test)]:
    print(f"{name} class balance:\n{y.value_counts(normalize=True).round(4)}")

pca_result = compare_pca_variants(
    x_tr_scaled, x_val_scaled, x_test_scaled, n_components="mle"
)
sklearn_pca = pca_result["sklearn_pca"]
x_tr_pca_sklearn = pca_result["x_tr_pca_sklearn"]
x_val_pca_sklearn = pca_result["x_val_pca_sklearn"]
x_test_pca_sklearn = pca_result["x_test_pca_sklearn"]
x_tr_pca_fast = pca_result["x_tr_pca_fast"]
x_val_pca_fast = pca_result["x_val_pca_fast"]
x_test_pca_fast = pca_result["x_test_pca_fast"]

x_tr_mc_scaled, x_val_mc_scaled, x_test_mc_scaled = (
    x_tr_scaled,
    x_val_scaled,
    x_test_scaled,
)

pca_mc = PCA(n_components="mle", svd_solver="full", random_state=RANDOM_SEED)
x_tr_mc_pca = pca_mc.fit_transform(x_tr_mc_scaled)
x_val_mc_pca = pca_mc.transform(x_val_mc_scaled)
x_test_mc_pca = pca_mc.transform(x_test_mc_scaled)

print(
    f"PCA components: {pca_mc.n_components_} | variance retained: {pca_mc.explained_variance_ratio_.sum():.4f}"
)


# %%
# =========================================================
# MODEL BUILDER
# =========================================================
def build_models(
    dt_params, rf_params, lr_params, random_state=RANDOM_SEED, class_weight=None
):
    dt = DecisionTreeClassifier(
        **dt_params,
        ccp_alpha=0.0,
        class_weight=class_weight,
        random_state=random_state,
    )

    rf = RandomForestClassifier(
        **rf_params,
        bootstrap=True,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=random_state,
    )

    lr = LogisticRegression(
        **lr_params,
        solver="lbfgs",
        class_weight=class_weight,
        random_state=random_state,
    )

    return {
        "Decision Tree": dt,
        "Random Forest": rf,
        "Logistic Regression": lr,
    }


# %%
# =========================================================
# HYPERPARAMETER TUNING
# =========================================================
MODEL_SPECS = [
    {
        "name": "Decision Tree",
        "estimator": DecisionTreeClassifier(ccp_alpha=0.0, random_state=RANDOM_SEED),
        "param_grid": {
            "max_depth": [None, 10, 15, 20, 25, 30],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 4, 8],
            "criterion": ["gini", "entropy"],
        },
        "n_iter": 15,
    },
    {
        "name": "Random Forest",
        "estimator": RandomForestClassifier(
            bootstrap=True, random_state=RANDOM_SEED, n_jobs=-1
        ),
        "param_grid": {
            "n_estimators": [100, 150],
            "max_depth": [None, 15, 25],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2"],
        },
        "n_iter": 6,
    },
    {
        "name": "Logistic Regression",
        "estimator": LogisticRegression(solver="lbfgs", random_state=RANDOM_SEED),
        "param_grid": {
            "C": np.logspace(-3, 3, 13),
            "max_iter": [500, 1000, 2000],
        },
        "n_iter": 8,
    },
]


def run_hyperparameter_tuning(x_tr_scaled, y_tr, scoring="f1", class_weight=None):
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    best_params = {}

    for spec in MODEL_SPECS:
        estimator = clone(spec["estimator"]).set_params(class_weight=class_weight)

        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=spec["param_grid"],
            n_iter=spec["n_iter"],
            cv=cv,
            scoring=scoring,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
        search.fit(x_tr_scaled, y_tr)
        best_params[spec["name"]] = search.best_params_
        print(
            f"{spec['name']}: best {scoring}={search.best_score_:.4f} | params={search.best_params_}"
        )

    return best_params


# ---- execution ----
best_params_binary = run_hyperparameter_tuning(
    x_tr_scaled, y_tr, scoring="f1", class_weight=None
)
dt_params = best_params_binary["Decision Tree"]
rf_params = best_params_binary["Random Forest"]
lr_params = best_params_binary["Logistic Regression"]

best_params_multiclass = run_hyperparameter_tuning(
    x_tr_mc_scaled, y_tr_mc, scoring="f1_macro", class_weight=None
)
dt_params_mc = best_params_multiclass["Decision Tree"]
rf_params_mc = best_params_multiclass["Random Forest"]
lr_params_mc = best_params_multiclass["Logistic Regression"]


# %%
# =========================================================
# PLOTTING HELPERS
# =========================================================
def save_fig(fig, name, dpi=300):
    fig.savefig(
        FIGURE_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight", facecolor="white"
    )


def find_optimal_threshold(y_true, y_proba):
    _, _, thresholds = roc_curve(y_true, y_proba)
    scores = [f1_score(y_true, (y_proba >= t).astype(int)) for t in thresholds]
    return thresholds[np.argmax(scores)]


def plot_feature_importance(model, feature_names, title, top_n=15, save_name=None):
    if hasattr(model, "feature_importances_"):
        scores = model.feature_importances_
    elif model.coef_.shape[0] == 1:
        scores = np.abs(model.coef_[0])
    else:
        scores = np.abs(model.coef_).mean(axis=0)

    feature_names = np.asarray(feature_names)
    top_idx = np.argsort(scores)[-top_n:][::-1]

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(
        feature_names[top_idx][::-1],
        scores[top_idx][::-1],
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_title(title)
    ax.set_xlabel("Importance")

    plt.tight_layout()
    if save_name:
        save_fig(fig, save_name)
    plt.show()


def plot_confusion_matrix(y_true, y_pred, labels, title, save_name=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
            )

    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    plt.tight_layout()
    if save_name:
        save_fig(fig, save_name)
    plt.show()


# %%
# =========================================================
# BINARY CLASSIFICATION PIPELINE
# =========================================================
MODEL_ORDER = ["Decision Tree", "Random Forest", "Logistic Regression"]


def run_binary_experiment(
    models, x_train, y_train, x_val, y_val, x_test, y_test, variant_label
):
    results = []
    for model_name, model in models.items():
        model.fit(x_train, y_train)

        y_val_proba = model.predict_proba(x_val)[:, 1]
        threshold = find_optimal_threshold(y_val, y_val_proba)

        y_test_proba = model.predict_proba(x_test)[:, 1]
        y_test_pred = (y_test_proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()

        print(
            f"{variant_label} | {model_name} | threshold={threshold:.4f} | acc={accuracy_score(y_test, y_test_pred):.4f}"
        )

        results.append(
            {
                "Model": model_name,
                "Variant": variant_label,
                "Threshold": threshold,
                "Accuracy": accuracy_score(y_test, y_test_pred),
                "Recall": recall_score(y_test, y_test_pred),
                "Precision": precision_score(y_test, y_test_pred),
                "F1": f1_score(y_test, y_test_pred),
                "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
                "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
                "_fitted_model": model,
                "_confusion_matrix": confusion_matrix(y_test, y_test_pred),
                "_y_test_proba": y_test_proba,
            }
        )
    return results


def run_binary_pipeline(
    x_tr_scaled,
    y_tr,
    x_val_scaled,
    y_val,
    x_test_scaled,
    y_test,
    x_tr_pca_sklearn,
    x_val_pca_sklearn,
    x_test_pca_sklearn,
    x_tr_pca_fast,
    x_val_pca_fast,
    x_test_pca_fast,
    dt_params,
    rf_params,
    lr_params,
):
    binary_variants = {
        "No PCA": (x_tr_scaled, x_val_scaled, x_test_scaled),
        "PCA (sklearn)": (x_tr_pca_sklearn, x_val_pca_sklearn, x_test_pca_sklearn),
        "PCA (fast)": (x_tr_pca_fast, x_val_pca_fast, x_test_pca_fast),
    }

    binary_results = []
    for variant_label, (x_train_v, x_val_v, x_test_v) in binary_variants.items():
        models = build_models(
            dt_params, rf_params, lr_params, random_state=RANDOM_SEED, class_weight=None
        )
        binary_results.extend(
            run_binary_experiment(
                models, x_train_v, y_tr, x_val_v, y_val, x_test_v, y_test, variant_label
            )
        )

    return binary_results


def print_accuracy_comparison(results_df):
    pivot = (
        results_df.pivot(index="Model", columns="Variant", values="Accuracy") * 100
    ).round(2)
    print(pivot.to_string())


def print_summary_statistics(results_df):
    for metric in ["Accuracy", "Precision", "Recall", "F1"]:
        print(f"\n{metric} by variant:")
        print(
            results_df.groupby("Variant")[metric]
            .agg(["mean", "std", "min", "max"])
            .round(4)
        )
        row = results_df.loc[results_df[metric].idxmax()]
        print(f"Best {metric}: {row['Model']} ({row['Variant']}) = {row[metric]:.4f}")


def plot_model_comparison_bars(results_df, metric="Accuracy"):
    pivot = results_df.pivot(index="Model", columns="Variant", values=metric)
    x_base = np.arange(len(pivot.index))
    width = 0.8 / len(pivot.columns)

    fig, ax = plt.subplots(figsize=(12, 6))
    for i, variant in enumerate(pivot.columns):
        bars = ax.bar(
            x_base + i * width,
            pivot[variant],
            width,
            label=variant,
            edgecolor="black",
            linewidth=0.5,
        )
        ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)

    ax.set_xticks(x_base + width * (len(pivot.columns) - 1) / 2)
    ax.set_xticklabels(pivot.index)
    ax.set_title(f"{metric} Comparison Across Feature Spaces")
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_roc_comparison(binary_results, y_test, variant="No PCA"):
    fig, ax = plt.subplots(figsize=(10, 8))
    for r in binary_results:
        if r["Variant"] != variant:
            continue
        fpr, tpr, _ = roc_curve(y_test, r["_y_test_proba"])
        ax.plot(fpr, tpr, lw=2.5, label=f"{r['Model']} (AUC={auc(fpr, tpr):.4f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves — {variant}")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


def extract_no_pca_models(binary_results):
    return {
        model_name: next(
            r["_fitted_model"]
            for r in binary_results
            if r["Variant"] == "No PCA" and r["Model"] == model_name
        )
        for model_name in MODEL_ORDER
    }


def plot_no_pca_confusion_matrices(binary_results, y_test):
    for model_name in MODEL_ORDER:
        r = next(
            r
            for r in binary_results
            if r["Variant"] == "No PCA" and r["Model"] == model_name
        )
        y_pred = (r["_y_test_proba"] >= r["Threshold"]).astype(int)
        plot_confusion_matrix(
            y_test,
            y_pred,
            ["Normal", "Attack"],
            f"Confusion Matrix — {model_name} (No PCA)",
        )


def run_binary_results_analysis(binary_results, x_all, y_test):
    results_df = pd.DataFrame(binary_results)

    print_accuracy_comparison(results_df)

    no_pca_models = extract_no_pca_models(binary_results)
    plot_feature_importance(
        no_pca_models["Decision Tree"],
        x_all.columns,
        "Top Features — Decision Tree (No PCA)",
    )
    plot_feature_importance(
        no_pca_models["Random Forest"],
        x_all.columns,
        "Top Features — Random Forest (No PCA)",
    )
    plot_feature_importance(
        no_pca_models["Logistic Regression"],
        x_all.columns,
        "Top Coefficients — Logistic Regression (No PCA)",
    )

    plot_no_pca_confusion_matrices(binary_results, y_test)

    for metric in ["Accuracy", "F1", "Recall"]:
        plot_model_comparison_bars(results_df, metric)

    plot_roc_comparison(binary_results, y_test, variant="No PCA")

    print_summary_statistics(results_df)

    return results_df, no_pca_models


# ---- execution ----
binary_results = run_binary_pipeline(
    x_tr_scaled,
    y_tr,
    x_val_scaled,
    y_val,
    x_test_scaled,
    y_test,
    x_tr_pca_sklearn,
    x_val_pca_sklearn,
    x_test_pca_sklearn,
    x_tr_pca_fast,
    x_val_pca_fast,
    x_test_pca_fast,
    dt_params,
    rf_params,
    lr_params,
)
results_df, no_pca_models = run_binary_results_analysis(binary_results, x_tr, y_test)


# %%
# =========================================================
# MULTICLASS CLASSIFICATION PIPELINE
# =========================================================
def check_multiclass_overfitting(
    models, x_tr, y_tr, x_val, y_val, x_test, y_test, class_names
):
    rows = []
    for model_name, model in models.items():
        model.fit(x_tr, y_tr)
        for split_name, x_split, y_split in [
            ("Train", x_tr, y_tr),
            ("Val", x_val, y_val),
            ("Test", x_test, y_test),
        ]:
            y_pred = model.predict(x_split)
            report = classification_report(
                y_split,
                y_pred,
                labels=range(len(class_names)),
                target_names=class_names,
                output_dict=True,
                zero_division=0,
            )
            rows.append(
                {
                    "Model": model_name,
                    "Split": split_name,
                    "Accuracy": accuracy_score(y_split, y_pred),
                    "Macro F1": report["macro avg"]["f1-score"],
                }
            )

    df = pd.DataFrame(rows)
    pivot = df.pivot(index="Model", columns="Split", values=["Accuracy", "Macro F1"])
    print(pivot.round(4).to_string())

    print("\nOverfitting gap (Train - Val):")
    gap = (
        df[df["Split"] == "Train"].set_index("Model")["Accuracy"]
        - df[df["Split"] == "Val"].set_index("Model")["Accuracy"]
    )
    print(gap.round(4))

    return df


def run_multiclass_experiment(
    models, x_train, y_train, x_test, y_test, class_names, variant_label
):
    results = []
    for model_name, model in models.items():
        model.fit(x_train, y_train)
        y_test_pred = model.predict(x_test)
        acc = accuracy_score(y_test, y_test_pred)
        report = classification_report(
            y_test,
            y_test_pred,
            labels=range(len(class_names)),
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        )

        print(
            f"{variant_label} | {model_name} | acc={acc:.4f} | macro_f1={report['macro avg']['f1-score']:.4f}"
        )

        results.append(
            {
                "Model": model_name,
                "Variant": variant_label,
                "Accuracy": acc,
                "Macro Precision": report["macro avg"]["precision"],
                "Macro Recall": report["macro avg"]["recall"],
                "Macro F1": report["macro avg"]["f1-score"],
                "Weighted Precision": report["weighted avg"]["precision"],
                "Weighted Recall": report["weighted avg"]["recall"],
                "Weighted F1": report["weighted avg"]["f1-score"],
                "_per_class_report": report,
                "_fitted_model": model,
                "_y_test_pred": y_test_pred,
            }
        )
    return results


def run_multiclass_pipeline(
    x_tr_mc_scaled,
    y_tr_mc,
    x_test_mc_scaled,
    y_test_mc,
    x_tr_mc_pca,
    x_test_mc_pca,
    dt_params_mc,
    rf_params_mc,
    lr_params_mc,
    class_names_with_unknown,
):
    multiclass_variants = {
        "No PCA": (x_tr_mc_scaled, x_test_mc_scaled),
        "PCA": (x_tr_mc_pca, x_test_mc_pca),
    }

    multiclass_results = []
    for variant_label, (x_train_v, x_test_v) in multiclass_variants.items():
        models = build_models(
            dt_params_mc,
            rf_params_mc,
            lr_params_mc,
            random_state=RANDOM_SEED,
            class_weight=None,
        )
        multiclass_results.extend(
            run_multiclass_experiment(
                models,
                x_train_v,
                y_tr_mc,
                x_test_v,
                y_test_mc,
                class_names_with_unknown,
                variant_label,
            )
        )

    return multiclass_results


def summarize_multiclass_results(multiclass_results):
    mc_results_df = pd.DataFrame(multiclass_results)

    accuracy_pivot = mc_results_df.pivot(
        index="Model", columns="Variant", values="Accuracy"
    ).round(4)
    print(accuracy_pivot)

    print("\nBest models by Weighted F1:")
    print(
        mc_results_df.nlargest(5, "Weighted F1")[
            ["Model", "Variant", "Weighted F1", "Accuracy", "Macro F1"]
        ]
    )

    print("\nMinority class alerts (recall < 0.50):")
    alerts = [
        {
            "Model": r["Model"],
            "Variant": r["Variant"],
            "Class": cls,
            "Recall": r["_per_class_report"][cls]["recall"],
        }
        for r in multiclass_results
        for cls in MINORITY_CLASSES
        if r["_per_class_report"][cls]["recall"] < 0.50
    ]
    print(pd.DataFrame(alerts) if alerts else "None")

    print("\nStats by variant:")
    print(
        mc_results_df.groupby("Variant")["Accuracy"]
        .agg(["mean", "std", "min", "max"])
        .round(4)
    )

    print("\nStats by model:")
    print(
        mc_results_df.groupby("Model")["Accuracy"]
        .agg(["mean", "std", "min", "max"])
        .round(4)
    )

    return mc_results_df


def build_per_class_report(multiclass_results, class_names_with_unknown):
    per_class_df = pd.DataFrame(
        [
            {
                "Model": r["Model"],
                "Variant": r["Variant"],
                "Class": cls,
                "Recall": r["_per_class_report"][cls]["recall"],
                "Precision": r["_per_class_report"][cls]["precision"],
                "F1": r["_per_class_report"][cls]["f1-score"],
                "Support": int(r["_per_class_report"][cls]["support"]),
            }
            for r in multiclass_results
            for cls in class_names_with_unknown
        ]
    )

    print("\nRecall by Model, Class, Variant:")
    print(
        per_class_df.pivot_table(
            index=["Model", "Class"], columns="Variant", values="Recall"
        ).round(4)
    )

    print("\nMinority class alert (recall < 0.50):")
    alerts = per_class_df[
        per_class_df["Class"].isin(MINORITY_CLASSES) & (per_class_df["Recall"] < 0.50)
    ]
    print(
        alerts[["Model", "Variant", "Class", "Recall", "Support"]]
        if not alerts.empty
        else "None"
    )

    print("\nBest minority class performance:")
    for cls in MINORITY_CLASSES:
        best = (
            per_class_df[per_class_df["Class"] == cls]
            .sort_values("Recall", ascending=False)
            .iloc[0]
        )
        print(f"{cls}: {best['Model']} ({best['Variant']}) recall={best['Recall']:.4f}")

    return per_class_df


# ---- execution ----
models_mc = build_models(
    dt_params_mc,
    rf_params_mc,
    lr_params_mc,
    random_state=RANDOM_SEED,
    class_weight=None,
)

overfit_check_df = check_multiclass_overfitting(
    models_mc,
    x_tr_mc_scaled,
    y_tr_mc,
    x_val_mc_scaled,
    y_val_mc,
    x_test_mc_scaled,
    y_test_mc,
    class_names_with_unknown,
)

multiclass_results = run_multiclass_pipeline(
    x_tr_mc_scaled,
    y_tr_mc,
    x_test_mc_scaled,
    y_test_mc,
    x_tr_mc_pca,
    x_test_mc_pca,
    dt_params_mc,
    rf_params_mc,
    lr_params_mc,
    class_names_with_unknown,
)

mc_results_df = summarize_multiclass_results(multiclass_results)
per_class_df = build_per_class_report(multiclass_results, class_names_with_unknown)

for model_name in MODEL_ORDER:
    r = next(
        r
        for r in multiclass_results
        if r["Variant"] == "No PCA" and r["Model"] == model_name
    )
    y_pred = r["_fitted_model"].predict(x_test_mc_scaled)
    plot_confusion_matrix(
        y_test_mc,
        y_pred,
        list(class_names_with_unknown),
        f"Confusion Matrix — {model_name} (Multiclass, No PCA)",
    )

print(
    f"\nBinary experiments: {len(binary_results)} | Multiclass experiments: {len(multiclass_results)}"
)


# %%
# =========================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# =========================================================
def compute_bootstrap_ci(
    y_true,
    y_pred,
    target_class,
    n_bootstrap=2000,
    ci_percentile=95.0,
    random_state=RANDOM_SEED,
    min_reliable_n=50,
):
    class_indices = np.nonzero(y_true == target_class)[0]
    if len(class_indices) == 0:
        return None

    if len(class_indices) < min_reliable_n:
        print(
            f"WARNING: only {len(class_indices)} samples for class={target_class}; CI unreliable, interpret with caution"
        )

    rng = np.random.default_rng(random_state)
    resampled_idx = rng.choice(
        class_indices, size=(n_bootstrap, len(class_indices)), replace=True
    )
    bootstrap_recalls = np.mean(y_pred[resampled_idx] == target_class, axis=1)

    alpha = 100 - ci_percentile
    ci_lower, ci_upper = np.percentile(bootstrap_recalls, [alpha / 2, 100 - alpha / 2])

    return {
        "n_samples": len(class_indices),
        "point_estimate": np.mean(y_pred[class_indices] == target_class),
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "ci_width": ci_upper - ci_lower,
    }


def run_bootstrap_ci_analysis(multiclass_results, y_test_mc, class_names_with_unknown):
    models_predictions = {
        (r["Model"], r["Variant"]): r["_y_test_pred"] for r in multiclass_results
    }

    minority_class_indices = {
        cls: list(class_names_with_unknown).index(cls) for cls in MINORITY_CLASSES
    }

    ci_rows = []
    for (model_name, variant), y_pred in models_predictions.items():
        for cls_name, cls_idx in minority_class_indices.items():
            ci = compute_bootstrap_ci(y_test_mc, y_pred, cls_idx)
            if ci is None:
                continue
            ci_rows.append(
                {"Model": model_name, "Variant": variant, "Class": cls_name, **ci}
            )

    ci_results_df = pd.DataFrame(ci_rows)
    print(ci_results_df.round(4).to_string(index=False))

    print("\nCI width by class:")
    print(
        ci_results_df.groupby("Class")["ci_width"].agg(["mean", "min", "max"]).round(4)
    )

    print("\nBest minority class recall:")
    for cls_name in minority_class_indices:
        best = ci_results_df[ci_results_df["Class"] == cls_name].loc[
            lambda d: d["point_estimate"].idxmax()
        ]
        print(
            f"{cls_name}: {best['Model']} ({best['Variant']}) = {best['point_estimate']:.4f} [{best['ci_lower']:.4f}, {best['ci_upper']:.4f}]"
        )

    return ci_results_df


# ---- execution ----
ci_results_df = run_bootstrap_ci_analysis(
    multiclass_results, y_test_mc, class_names_with_unknown
)


# %%
# =========================================================
# PCA-RELATED PLOTS + SENSITIVITY SWEEP
# =========================================================
def plot_before_after_pca(x_tr_mc_scaled, y_tr_mc, class_names_with_unknown):
    pca_2d = PCA(n_components=2, svd_solver="full", random_state=RANDOM_SEED)
    x_tr_mc_2d = pca_2d.fit_transform(x_tr_mc_scaled)

    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
    for cls_idx, cls_name in enumerate(class_names_with_unknown):
        mask = y_tr_mc == cls_idx
        axes[0].scatter(
            x_tr_mc_scaled[mask, 0],
            x_tr_mc_scaled[mask, 1],
            s=15,
            alpha=0.7,
            edgecolor="black",
            linewidth=0.3,
            label=cls_name,
        )
        axes[1].scatter(
            x_tr_mc_2d[mask, 0],
            x_tr_mc_2d[mask, 1],
            s=15,
            alpha=0.7,
            edgecolor="black",
            linewidth=0.3,
            label=cls_name,
        )

    axes[0].set_title("Before PCA")
    axes[0].set_xlabel("Standardized Feature 1")
    axes[0].set_ylabel("Standardized Feature 2")
    axes[1].set_title("After PCA")
    axes[1].set_xlabel("PC1")
    axes[1].set_ylabel("PC2")
    axes[1].legend(loc="upper right")
    fig.suptitle("Before vs After PCA Transformation (Training Data)")
    fig.tight_layout()
    save_fig(fig, "v0_before_after_pca")
    plt.show()


def plot_scree(x_tr_scaled, sklearn_pca):
    pca_full = PCA(svd_solver="full", random_state=RANDOM_SEED).fit(x_tr_scaled)
    cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)
    n_range = np.arange(1, len(cumulative_var) + 1)

    fig, ax1 = plt.subplots(figsize=FIGSIZE)
    ax1.bar(
        n_range,
        pca_full.explained_variance_ratio_,
        alpha=0.5,
        label="Individual Variance",
    )
    ax2 = ax1.twinx()
    ax2.plot(
        n_range,
        cumulative_var,
        marker="o",
        markersize=3,
        linewidth=2,
        color="orange",
        label="Cumulative Variance",
    )
    ax2.axvline(
        sklearn_pca.n_components_,
        linestyle="--",
        color="gray",
        label=f"MLE n={sklearn_pca.n_components_}",
    )
    ax1.set_xlabel("Principal Component")
    ax1.set_ylabel("Individual Explained Variance")
    ax2.set_ylabel("Cumulative Explained Variance")
    ax1.set_title("PCA Scree Plot")
    fig.legend(loc="center right")
    fig.tight_layout()
    save_fig(fig, "v1_scree_plot")
    plt.show()


def plot_roc_all_variants(
    binary_results, y_test, x_test_scaled, x_test_pca_sklearn, x_test_pca_fast
):
    variant_test_data = {
        "No PCA": x_test_scaled,
        "PCA (sklearn)": x_test_pca_sklearn,
        "PCA (fast)": x_test_pca_fast,
    }

    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, sharey=True)
    for ax, (variant_label, x_te_v) in zip(axes, variant_test_data.items()):
        for r in binary_results:
            if r["Variant"] != variant_label:
                continue
            fpr, tpr, _ = roc_curve(
                y_test, r["_fitted_model"].predict_proba(x_te_v)[:, 1]
            )
            ax.plot(
                fpr, tpr, linewidth=2, label=f"{r['Model']} (AUC={auc(fpr, tpr):.4f})"
            )
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Chance")
        ax.set_xlabel("False Positive Rate")
        ax.set_title(variant_label)
        ax.legend(loc="lower right", fontsize=9)

    axes[0].set_ylabel("True Positive Rate")
    fig.suptitle("ROC Curves — Binary Classification")
    fig.tight_layout()
    save_fig(fig, "v2_roc_curves")
    plt.show()


def per_class_recall(model, x_te, y_te, class_names_with_unknown):
    report = classification_report(
        y_te,
        model.predict(x_te),
        labels=range(len(class_names_with_unknown)),
        target_names=class_names_with_unknown,
        output_dict=True,
        zero_division=0,
    )
    return (
        report["U2R"]["recall"],
        report["R2L"]["recall"],
        report["macro avg"]["f1-score"],
    )


def fit_variant_and_get_recalls(
    x_train,
    x_test,
    y_train,
    y_test,
    dt_params_mc,
    rf_params_mc,
    lr_params_mc,
    class_names_with_unknown,
):
    models = build_models(
        dt_params_mc,
        rf_params_mc,
        lr_params_mc,
        random_state=RANDOM_SEED,
        class_weight=None,
    )
    row = {}
    for prefix, model in zip(["DT", "RF", "LR"], models.values()):
        model.fit(x_train, y_train)
        u2r, r2l, macro_f1 = per_class_recall(
            model, x_test, y_test, class_names_with_unknown
        )
        row.update(
            {f"{prefix}_U2R": u2r, f"{prefix}_R2L": r2l, f"{prefix}_MacroF1": macro_f1}
        )
    return row


def plot_pca_sensitivity_sweep(
    x_tr_mc_scaled,
    x_test_mc_scaled,
    y_tr_mc,
    y_test_mc,
    dt_params_mc,
    rf_params_mc,
    lr_params_mc,
    class_names_with_unknown,
):
    pca_full_mc = PCA(svd_solver="full", random_state=RANDOM_SEED).fit(x_tr_mc_scaled)
    cum_var_mc = np.cumsum(pca_full_mc.explained_variance_ratio_)

    sweep_rows = [
        {
            "n_components": x_tr_mc_scaled.shape[1],
            **fit_variant_and_get_recalls(
                x_tr_mc_scaled,
                x_test_mc_scaled,
                y_tr_mc,
                y_test_mc,
                dt_params_mc,
                rf_params_mc,
                lr_params_mc,
                class_names_with_unknown,
            ),
        }
    ]

    for var_target in [0.999, 0.99, 0.98, 0.95, 0.90, 0.85]:
        n_comp = int(np.argmax(cum_var_mc >= var_target) + 1)
        pca_v = PCA(n_components=n_comp, svd_solver="full", random_state=RANDOM_SEED)
        x_tr_v = pca_v.fit_transform(x_tr_mc_scaled)
        x_te_v = pca_v.transform(x_test_mc_scaled)
        sweep_rows.append(
            {
                "n_components": n_comp,
                **fit_variant_and_get_recalls(
                    x_tr_v,
                    x_te_v,
                    y_tr_mc,
                    y_test_mc,
                    dt_params_mc,
                    rf_params_mc,
                    lr_params_mc,
                    class_names_with_unknown,
                ),
            }
        )

    sweep_df = pd.DataFrame(sweep_rows).sort_values("n_components")
    print(sweep_df.to_string(index=False))

    metric_specs = [
        ("U2R", "U2R Recall"),
        ("R2L", "R2L Recall"),
        ("MacroF1", "Macro-F1"),
    ]
    model_specs = [
        ("DT", "Decision Tree"),
        ("RF", "Random Forest"),
        ("LR", "Logistic Regression"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE)
    for ax, (metric_key, metric_label) in zip(axes, metric_specs):
        for prefix, label in model_specs:
            ax.plot(
                sweep_df["n_components"],
                sweep_df[f"{prefix}_{metric_key}"],
                marker="o",
                label=label,
                linewidth=2,
            )
        ax.set_xlabel("Number of PCA Components")
        ax.set_ylabel(metric_label)
        ax.set_title(f"{metric_label} vs PCA Components")
        ax.legend()
        ax.invert_xaxis()

    fig.suptitle("PCA Sensitivity Sweep — Multiclass Task")
    fig.tight_layout()
    save_fig(fig, "v3_pca_sensitivity_sweep")
    plt.show()

    return sweep_df


# ---- execution ----
plot_before_after_pca(x_tr_mc_scaled, y_tr_mc, class_names_with_unknown)
plot_scree(x_tr_scaled, sklearn_pca)
plot_roc_all_variants(
    binary_results, y_test, x_test_scaled, x_test_pca_sklearn, x_test_pca_fast
)
sweep_df = plot_pca_sensitivity_sweep(
    x_tr_mc_scaled,
    x_test_mc_scaled,
    y_tr_mc,
    y_test_mc,
    dt_params_mc,
    rf_params_mc,
    lr_params_mc,
    class_names_with_unknown,
)


# %%
# =========================================================
# FINAL SUMMARY PLOTS
# =========================================================
def plot_bootstrap_ci_errorbars(ci_results_df):
    x_base = np.arange(len(MODEL_ORDER))
    width = 0.3

    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
    for ax, cls_name in zip(axes, MINORITY_CLASSES):
        subset = ci_results_df[ci_results_df["Class"] == cls_name]
        for i, variant in enumerate(["No PCA", "PCA"]):
            row = (
                subset[subset["Variant"] == variant].set_index("Model").loc[MODEL_ORDER]
            )
            points = row["point_estimate"].to_numpy()
            err = [
                points - row["ci_lower"].to_numpy(),
                row["ci_upper"].to_numpy() - points,
            ]
            ax.errorbar(
                x_base + (i - 0.5) * width,
                points,
                yerr=err,
                fmt="o",
                markersize=8,
                capsize=6,
                label=variant,
            )
        ax.set_xticks(x_base)
        ax.set_xticklabels(MODEL_ORDER)
        ax.set_ylabel("Recall")
        ax.set_title(f"{cls_name} Recall — 95% Bootstrap CI")
        ax.set_ylim(0, 1.05)
        ax.legend()

    fig.suptitle("Bootstrap 95% Confidence Intervals — No PCA vs PCA")
    fig.tight_layout()
    save_fig(fig, "v4_bootstrap_ci")
    plt.show()


def plot_class_distribution(y_tr_mc, y_test_mc, class_names_with_unknown):
    dist_df = (
        pd.DataFrame(
            {
                "Train": pd.Series(y_tr_mc)
                .map(dict(enumerate(class_names_with_unknown)))
                .value_counts(),
                "Test": pd.Series(y_test_mc)
                .map(dict(enumerate(class_names_with_unknown)))
                .value_counts(),
            }
        )
        .reindex(class_names_with_unknown)
        .fillna(0)
        .astype(int)
    )

    x_base = np.arange(len(dist_df))
    width = 0.35

    fig, ax = plt.subplots(figsize=FIGSIZE)
    bars_train = ax.bar(x_base - width / 2, dist_df["Train"], width, label="Train")
    bars_test = ax.bar(x_base + width / 2, dist_df["Test"], width, label="Test")
    ax.set_yscale("log")
    ax.set_xticks(x_base)
    ax.set_xticklabels(dist_df.index)
    ax.set_ylabel("Sample Count (log scale)")
    ax.set_title("Class Distribution — Train vs Test (Log Scale)")
    ax.bar_label(bars_train, fontsize=8, padding=2)
    ax.bar_label(bars_test, fontsize=8, padding=2)
    ax.legend()

    fig.tight_layout()
    save_fig(fig, "v5_class_distribution")
    plt.show()

    print(dist_df.to_string())
    return dist_df


def plot_summary_grouped(multiclass_results):
    summary_df = pd.DataFrame(
        [
            {
                "Model": r["Model"],
                "Variant": r["Variant"],
                "Accuracy": r["Accuracy"],
                "Macro F1": r["Macro F1"],
                **{
                    f"{cls} Recall": r["_per_class_report"][cls]["recall"]
                    for cls in MINORITY_CLASSES
                },
            }
            for r in multiclass_results
        ]
    )
    metrics = ["Accuracy", "Macro F1"] + [f"{cls} Recall" for cls in MINORITY_CLASSES]

    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE)
    for ax, metric in zip(axes.flatten(), metrics):
        x_base = np.arange(len(MODEL_ORDER))
        bar_width = 0.35
        for i, variant in enumerate(["No PCA", "PCA"]):
            values = [
                summary_df[
                    (summary_df["Model"] == m) & (summary_df["Variant"] == variant)
                ][metric].iloc[0]
                for m in MODEL_ORDER
            ]
            bars = ax.bar(
                x_base + (i - 0.5) * bar_width,
                values,
                bar_width,
                label=variant,
                edgecolor="black",
                linewidth=0.5,
            )
            ax.bar_label(bars, fmt="%.3f", fontsize=9, padding=2)
        ax.set_xticks(x_base)
        ax.set_xticklabels(MODEL_ORDER)
        ax.set_title(metric)
        ax.set_ylim(0, 1.05)

    axes[0, 0].legend()
    fig.suptitle("No PCA vs PCA — Summary Across Key Metrics")
    fig.tight_layout()
    save_fig(fig, "v6_summary_grouped")
    plt.show()

    return summary_df


# ---- execution ----
plot_bootstrap_ci_errorbars(ci_results_df)
dist_df = plot_class_distribution(y_tr_mc, y_test_mc, class_names_with_unknown)
summary_df = plot_summary_grouped(multiclass_results)

print(
    f"\nBinary experiments: {len(binary_results)} | Multiclass experiments: {len(multiclass_results)}"
)

In [ ]:
# %%
# =========================================================
# ABLATION RUNNER — set ABLATION and run this cell/script
# =========================================================
ABLATION = "pca_kernel"
# options:
# "imbalance"          -> Baseline vs SMOTE vs class_weight='balanced'
# "unknown_class"      -> keep Unknown class vs drop novel rows
# "scaling"            -> StandardScaler vs MinMaxScaler vs NoScaling
# "encoding"           -> OneHot vs Ordinal categorical encoding
# "threshold"          -> fixed 0.5 threshold vs optimized threshold (binary)
# "known_vs_novel"     -> test performance on known-only vs novel-only attack rows
# "feature_selection"  -> top-K important features vs all features
# "pca_kernel"         -> Linear PCA vs Kernel PCA (multiclass)

print(f"Running ablation: {ABLATION}")

# ---------------------------------------------------------
# 1. IMBALANCE HANDLING ABLATION
# ---------------------------------------------------------
if ABLATION == "imbalance":
    from imblearn.over_sampling import SMOTE

    def apply_smote_ablation(x_train, y_train, random_state=RANDOM_SEED):
        smote = SMOTE(
            sampling_strategy="not majority",
            random_state=random_state,
            k_neighbors=5,
        )
        x_resampled, y_resampled = smote.fit_resample(x_train, y_train)
        print(f"SMOTE: {len(y_train):,} -> {len(y_resampled):,} samples")
        return x_resampled, y_resampled

    def per_class_recall_ablation(model, x_te, y_te, class_names_arr):
        report = classification_report(
            y_te,
            model.predict(x_te),
            labels=range(len(class_names_arr)),
            target_names=class_names_arr,
            output_dict=True,
            zero_division=0,
        )
        return (
            report["U2R"]["recall"],
            report["R2L"]["recall"],
            report["macro avg"]["f1-score"],
        )

    imbalance_variants = {}
    imbalance_variants["Baseline"] = (x_tr_mc_scaled, y_tr_mc, None)

    x_tr_mc_smote, y_tr_mc_smote = apply_smote_ablation(x_tr_mc_scaled, y_tr_mc)
    imbalance_variants["SMOTE"] = (x_tr_mc_smote, y_tr_mc_smote, None)

    imbalance_variants["ClassWeight"] = (x_tr_mc_scaled, y_tr_mc, "balanced")

    imbalance_rows = []
    for variant_label, (x_train_v, y_train_v, cw) in imbalance_variants.items():
        models_v = build_models(
            dt_params_mc,
            rf_params_mc,
            lr_params_mc,
            random_state=RANDOM_SEED,
            class_weight=cw,
        )
        for model_name, model in models_v.items():
            model.fit(x_train_v, y_train_v)
            u2r, r2l, macro_f1 = per_class_recall_ablation(
                model, x_test_mc_scaled, y_test_mc, class_names_with_unknown
            )
            acc = accuracy_score(y_test_mc, model.predict(x_test_mc_scaled))
            imbalance_rows.append(
                {
                    "Variant": variant_label,
                    "Model": model_name,
                    "Accuracy": acc,
                    "U2R_Recall": u2r,
                    "R2L_Recall": r2l,
                    "MacroF1": macro_f1,
                }
            )

    imbalance_ablation_df = pd.DataFrame(imbalance_rows)
    print(imbalance_ablation_df.round(4).to_string(index=False))


# ---------------------------------------------------------
# 2. UNKNOWN CLASS ABLATION (keep vs drop)
# ---------------------------------------------------------
elif ABLATION == "unknown_class":
    novel_mask_test = (y_test_mc == len(class_names)).to_numpy()
    n_novel = novel_mask_test.sum()
    print(f"Novel/unseen rows in test: {n_novel} / {len(y_test_mc)}")

    x_test_mc_known = x_test_mc_scaled[~novel_mask_test]
    y_test_mc_known = y_test_mc[~novel_mask_test]

    unknown_variants = {
        "Keep (with Unknown)": (x_test_mc_scaled, y_test_mc),
        "Drop (novel removed)": (x_test_mc_known, y_test_mc_known),
    }

    unknown_rows = []
    for variant_label, (x_te, y_te) in unknown_variants.items():
        models_v = build_models(
            dt_params_mc,
            rf_params_mc,
            lr_params_mc,
            random_state=RANDOM_SEED,
            class_weight=None,
        )
        for model_name, model in models_v.items():
            model.fit(x_tr_mc_scaled, y_tr_mc)
            y_pred = model.predict(x_te)
            labels_present = sorted(set(y_te) | set(y_pred))
            report = classification_report(
                y_te,
                y_pred,
                labels=labels_present,
                target_names=[class_names_with_unknown[i] for i in labels_present],
                output_dict=True,
                zero_division=0,
            )
            unknown_rows.append(
                {
                    "Variant": variant_label,
                    "Model": model_name,
                    "Accuracy": accuracy_score(y_te, y_pred),
                    "MacroF1": report["macro avg"]["f1-score"],
                }
            )

    unknown_ablation_df = pd.DataFrame(unknown_rows)
    print(unknown_ablation_df.round(4).to_string(index=False))


# ---------------------------------------------------------
# 3. SCALING ABLATION
# ---------------------------------------------------------
elif ABLATION == "scaling":
    import warnings
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.preprocessing import MinMaxScaler

    scaler_variants = {
        "StandardScaler": StandardScaler(),
        "MinMaxScaler": MinMaxScaler(),
        "NoScaling": None,
    }

    scaling_rows = []
    for scaler_name, scaler_obj in scaler_variants.items():
        if scaler_obj is None:
            x_tr_s = x_tr.to_numpy(dtype=np.float64)
            x_test_s = x_test.to_numpy(dtype=np.float64)
        else:
            x_tr_s = scaler_obj.fit_transform(x_tr.astype(np.float64))
            x_test_s = scaler_obj.transform(x_test.astype(np.float64))

        lr_params_scaling = {**lr_params, "max_iter": 5000}

        models_v = build_models(
            dt_params, rf_params, lr_params_scaling, random_state=RANDOM_SEED
        )
        for model_name, model in models_v.items():
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=ConvergenceWarning)
                model.fit(x_tr_s, y_tr)
            y_pred = model.predict(x_test_s)
            scaling_rows.append(
                {
                    "Scaler": scaler_name,
                    "Model": model_name,
                    "Accuracy": accuracy_score(y_test, y_pred),
                    "F1": f1_score(y_test, y_pred),
                    "LR_Converged": scaler_name != "NoScaling",  # informational flag
                }
            )

    scaling_ablation_df = pd.DataFrame(scaling_rows)
    print(scaling_ablation_df.round(4).to_string(index=False))


# ---------------------------------------------------------
# 4. CATEGORICAL ENCODING ABLATION
# ---------------------------------------------------------
elif ABLATION == "encoding":
    from sklearn.preprocessing import OrdinalEncoder

    def transform_with_encoder_ablation(df, cat_encoder, encoding_type):
        df = df.copy()
        cat_encoded = cat_encoder.transform(df[CATEGORICAL_COLUMNS])
        if encoding_type == "onehot":
            cat_cols = cat_encoder.get_feature_names_out(CATEGORICAL_COLUMNS)
        else:
            cat_cols = CATEGORICAL_COLUMNS
        cat_df = pd.DataFrame(cat_encoded, columns=cat_cols, index=df.index)
        numeric_df = df.drop(columns=CATEGORICAL_COLUMNS + ["target", "category"])
        x_out = pd.concat([numeric_df, cat_df], axis=1)
        y_out = df["target"]
        return x_out, y_out

    encoding_rows = []
    for encoding_type in ["onehot", "ordinal"]:
        if encoding_type == "onehot":
            cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        else:
            cat_encoder = OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1
            )
        cat_encoder.fit(train_df[CATEGORICAL_COLUMNS])

        x_tr_e, y_tr_e = transform_with_encoder_ablation(
            train_df, cat_encoder, encoding_type
        )
        x_test_e, y_test_e = transform_with_encoder_ablation(
            test_df, cat_encoder, encoding_type
        )

        scaler_e = StandardScaler()
        x_tr_e_scaled = scaler_e.fit_transform(x_tr_e.astype(np.float64))
        x_test_e_scaled = scaler_e.transform(x_test_e.astype(np.float64))

        models_v = build_models(
            dt_params, rf_params, lr_params, random_state=RANDOM_SEED
        )
        for model_name, model in models_v.items():
            model.fit(x_tr_e_scaled, y_tr_e)
            y_pred = model.predict(x_test_e_scaled)
            encoding_rows.append(
                {
                    "Encoding": encoding_type,
                    "Model": model_name,
                    "Accuracy": accuracy_score(y_test_e, y_pred),
                    "F1": f1_score(y_test_e, y_pred),
                }
            )

    encoding_ablation_df = pd.DataFrame(encoding_rows)
    print(encoding_ablation_df.round(4).to_string(index=False))


# ---------------------------------------------------------
# 5. THRESHOLD ABLATION (binary) — fixed 0.5 vs optimized
# ---------------------------------------------------------
elif ABLATION == "threshold":
    threshold_rows = []
    models_v = build_models(dt_params, rf_params, lr_params, random_state=RANDOM_SEED)
    for model_name, model in models_v.items():
        model.fit(x_tr_scaled, y_tr)
        y_val_proba = model.predict_proba(x_val_scaled)[:, 1]
        y_test_proba = model.predict_proba(x_test_scaled)[:, 1]

        for label, thr in [
            ("Fixed 0.5", 0.5),
            ("Optimized", find_optimal_threshold(y_val, y_val_proba)),
        ]:
            y_pred = (y_test_proba >= thr).astype(int)
            threshold_rows.append(
                {
                    "Model": model_name,
                    "ThresholdType": label,
                    "Threshold": thr,
                    "Accuracy": accuracy_score(y_test, y_pred),
                    "Precision": precision_score(y_test, y_pred),
                    "Recall": recall_score(y_test, y_pred),
                    "F1": f1_score(y_test, y_pred),
                }
            )

    threshold_ablation_df = pd.DataFrame(threshold_rows)
    print(threshold_ablation_df.round(4).to_string(index=False))


# ---------------------------------------------------------
# 6. KNOWN vs NOVEL TEST EVALUATION (generalization check)
# ---------------------------------------------------------
elif ABLATION == "known_vs_novel":
    novel_categories = set(test_df["category"].unique()) - set(
        train_df["category"].unique()
    )
    print(f"Novel categories present only in test: {novel_categories}")

    novel_mask = test_df["category"].isin(novel_categories).to_numpy()
    print(f"Novel rows: {novel_mask.sum()} / {len(novel_mask)}")

    known_novel_rows = []
    models_v = build_models(dt_params, rf_params, lr_params, random_state=RANDOM_SEED)
    for model_name, model in models_v.items():
        model.fit(x_tr_scaled, y_tr)
        y_pred_all = model.predict(x_test_scaled)
        y_test_arr = y_test.to_numpy()

        for label, mask in [
            ("Known-only", ~novel_mask),
            ("Novel-only", novel_mask),
            ("All", np.ones_like(novel_mask, dtype=bool)),
        ]:
            if mask.sum() == 0:
                continue
            known_novel_rows.append(
                {
                    "Model": model_name,
                    "Subset": label,
                    "N": int(mask.sum()),
                    "Accuracy": accuracy_score(y_test_arr[mask], y_pred_all[mask]),
                    "Recall": recall_score(
                        y_test_arr[mask], y_pred_all[mask], zero_division=0
                    ),
                }
            )

    known_novel_df = pd.DataFrame(known_novel_rows)
    print(known_novel_df.round(4).to_string(index=False))


# ---------------------------------------------------------
# 7. FEATURE SELECTION ABLATION (Top-K vs All)
# ---------------------------------------------------------
elif ABLATION == "feature_selection":
    rf_full = RandomForestClassifier(**rf_params, random_state=RANDOM_SEED, n_jobs=-1)
    rf_full.fit(x_tr_scaled, y_tr)
    importances = rf_full.feature_importances_
    feature_names_arr = np.asarray(x_tr.columns)

    feature_selection_rows = []
    for k in [10, 20, 30, len(feature_names_arr)]:
        top_idx = np.argsort(importances)[-k:]
        x_tr_k = x_tr_scaled[:, top_idx]
        x_test_k = x_test_scaled[:, top_idx]

        models_v = build_models(
            dt_params, rf_params, lr_params, random_state=RANDOM_SEED
        )
        for model_name, model in models_v.items():
            model.fit(x_tr_k, y_tr)
            y_pred = model.predict(x_test_k)
            feature_selection_rows.append(
                {
                    "TopK": k,
                    "Model": model_name,
                    "Accuracy": accuracy_score(y_test, y_pred),
                    "F1": f1_score(y_test, y_pred),
                }
            )

    feature_selection_df = pd.DataFrame(feature_selection_rows)
    print(feature_selection_df.round(4).to_string(index=False))


# ---------------------------------------------------------
# 8. LINEAR PCA vs KERNEL PCA (multiclass)
# ---------------------------------------------------------
elif ABLATION == "pca_kernel":
    from sklearn.decomposition import KernelPCA

    n_comp = pca_mc.n_components_

    # KernelPCA-r kernel matrix N x N size hoy, tai puro train set (~100k rows)
    # e fit kora possible na (75+ GB memory lagbe). Subsample kore fit kora hocche.
    KPCA_FIT_SAMPLE_SIZE = 5000

    rng = np.random.default_rng(RANDOM_SEED)
    if x_tr_mc_scaled.shape[0] > KPCA_FIT_SAMPLE_SIZE:
        sample_idx = rng.choice(
            x_tr_mc_scaled.shape[0], size=KPCA_FIT_SAMPLE_SIZE, replace=False
        )
        x_tr_mc_kpca_fit = x_tr_mc_scaled[sample_idx]
        y_tr_mc_kpca_fit = (
            y_tr_mc.to_numpy()[sample_idx]
            if hasattr(y_tr_mc, "to_numpy")
            else y_tr_mc[sample_idx]
        )
    else:
        x_tr_mc_kpca_fit = x_tr_mc_scaled
        y_tr_mc_kpca_fit = y_tr_mc

    print(
        f"KernelPCA fit sample size: {x_tr_mc_kpca_fit.shape[0]} (out of {x_tr_mc_scaled.shape[0]})"
    )

    pca_kernel_variants = {}

    pca_lin = PCA(n_components=n_comp, svd_solver="full", random_state=RANDOM_SEED)
    pca_kernel_variants["Linear PCA"] = (
        pca_lin.fit_transform(x_tr_mc_scaled),
        y_tr_mc,
        pca_lin.transform(x_test_mc_scaled),
    )

    kpca = KernelPCA(
        n_components=n_comp, kernel="rbf", random_state=RANDOM_SEED, n_jobs=-1
    )
    x_tr_kpca = kpca.fit_transform(x_tr_mc_kpca_fit)  # subsample-e fit + transform
    x_test_kpca = kpca.transform(
        x_test_mc_scaled
    )  # eta already fitted kpca-r upor transform
    pca_kernel_variants["Kernel PCA (rbf)"] = (
        x_tr_kpca,
        y_tr_mc_kpca_fit,
        x_test_kpca,
    )

    pca_kernel_rows = []
    for variant_label, (x_tr_v, y_tr_v, x_test_v) in pca_kernel_variants.items():
        models_v = build_models(
            dt_params_mc,
            rf_params_mc,
            lr_params_mc,
            random_state=RANDOM_SEED,
            class_weight=None,
        )
        for model_name, model in models_v.items():
            model.fit(x_tr_v, y_tr_v)
            y_pred = model.predict(x_test_v)
            report = classification_report(
                y_test_mc,
                y_pred,
                labels=range(len(class_names_with_unknown)),
                target_names=class_names_with_unknown,
                output_dict=True,
                zero_division=0,
            )
            pca_kernel_rows.append(
                {
                    "Variant": variant_label,
                    "Model": model_name,
                    "TrainSize": x_tr_v.shape[0],
                    "Accuracy": accuracy_score(y_test_mc, y_pred),
                    "MacroF1": report["macro avg"]["f1-score"],
                }
            )

    pca_kernel_df = pd.DataFrame(pca_kernel_rows)
    print(pca_kernel_df.round(4).to_string(index=False))

else:
    raise ValueError(f"Unknown ABLATION: {ABLATION}")